# SC3021 Data Science Fundamentals: Week 9 Tutorial
## Data Management (Part 2) - SQL and Database Design
**Material in this notebook developed by Joel Quek**

Welcome to the interactive portion of this week's tutorial! While conceptual modeling (drawing ER diagrams) is the first step of database design, data scientists may spend a good amount of their time interacting with relational databases as well, very often, pulling the neccersary information to work with dataframes.

Today, we will use SQLite and Python's `pandas` library to actually build and query the databases described in your tutorial sheet.




In [ ]:
import sqlite3
import pandas as pd

# Create an in-memory SQLite database
conn = sqlite3.connect(':memory:')

# Helper function to run SQL queries and display them nicely as pandas DataFrames
def run_query(query):
    try:
        return pd.read_sql_query(query, conn)
    except Exception as e:
        return f"SQL Error: {e}"

print("SQLite Database Connected! Ready to execute SQL.")

SQLite Database Connected! Ready to execute SQL.


---
### Part 1: Modeling Q2 - The 1:N Relationship
The tutorial asks us to write SQL code declaring the tables necessary to implement a logical data model where a coach trains multiple players, but a player is trained by only one coach.

Let's write the Data Definition Language (DDL) to create these tables, using primary keys and foreign key constraints. Then, we'll insert some mock data using Data Manipulation Language (DML).

**Please note that I only included DROP TABLE IF EXISTS for your convenience if you want to rerun the cell, it should not actually be part of the solution**

In [ ]:
# Create the coaches and players tables

conn.executescript('''
DROP TABLE IF EXISTS trains;
DROP TABLE IF EXISTS players;
DROP TABLE IF EXISTS coaches;

''')


conn.executescript('''
CREATE TABLE coaches (
    coachID INTEGER NOT NULL PRIMARY KEY,
    name VARCHAR(100) NOT NULL,
    experienceLevel VARCHAR(50)
);

CREATE TABLE players (
    playerID INTEGER NOT NULL PRIMARY KEY,
    name VARCHAR(100) NOT NULL,
    age INTEGER,
    position VARCHAR(50),
    coachRef INTEGER REFERENCES coaches(coachID)
);

-- Insert mock data
INSERT INTO coaches VALUES (1, 'Coach Dan', 'Senior'), (2, 'Coach Don', 'Junior');
INSERT INTO players VALUES (101, 'Tim', 20, 'Forward', 2), (102, 'Tom', 22, 'Defender', 1), (103, 'Sam', 19, 'Goalie', 1);
''')

print("Tables 'coaches' and 'players' created successfully.")
display(run_query("SELECT * FROM players;"))

Tables 'coaches' and 'players' created successfully.


,playerID,name,age,position,coachRef
0,101,Tim,20,Forward,2
1,102,Tom,22,Defender,1
2,103,Sam,19,Goalie,1


---
### What if this changes to a M:N context?
Look at the `players` table above. In reality, a player might have a Head Coach, a Fitness Coach, and a Dietitian.

How best to do this?

**The Fix:** Step 2 of the ER-to-Relational algorithm tells us that for each relationship type, we must create a relation (table) containing the key attributes of the two related entities. Let's drop our flawed table and build a proper association table!

In [ ]:
# Drop the poorly designed players table
conn.executescript('''
DROP TABLE players;

-- Recreate players without the foreign key
CREATE TABLE players (
    playerID INTEGER NOT NULL PRIMARY KEY,
    name VARCHAR(100) NOT NULL,
    age INTEGER,
    position VARCHAR(50)
);

-- Create the association table for the M:N relationship
CREATE TABLE trains (
    coachID INTEGER REFERENCES coaches(coachID),
    playerID INTEGER REFERENCES players(playerID),
    hours_per_week INTEGER,
    PRIMARY KEY (coachID, playerID)
);

-- Insert data into the new schema
INSERT INTO players VALUES (101, 'Tim', 20, 'Forward'), (102, 'Tom', 22, 'Defender');
-- Coach 1 trains Tim (10 hours) and Tom (15 hours). Coach 2 trains Tim (5 hours).
INSERT INTO trains VALUES (1, 101, 10), (2, 101, 5), (1, 102, 15);
''')

print("Association table 'trains' created to handle the M:N relationship!")
display(run_query("SELECT * FROM trains;"))

Association table 'trains' created to handle the M:N relationship!


,coachID,playerID,hours_per_week
0,1,101,10
1,2,101,5
2,1,102,15


In [ ]:
trains = run_query("SELECT * FROM trains")
players = run_query("SELECT * FROM players")
coaches = run_query("SELECT * FROM coaches")

In [ ]:
trains

,coachID,playerID,hours_per_week
0,1,101,10
1,2,101,5
2,1,102,15


In [ ]:
players

,playerID,name,age,position
0,101,Tim,20,Forward
1,102,Tom,22,Defender


In [ ]:
coaches

,coachID,name,experienceLevel
0,1,Coach Dan,Senior
1,2,Coach Don,Junior


---
### Part 2: SQL Q1 - The Cartesian Product Trap
The tutorial provides two tables, `Coach` and `Skater`, and asks us to predict the output of a query.

Notice that the `SELECT` query has two tuple variables (`Coach C, Skater S`) in the `FROM` clause, but **no `WHERE` clause** linking them. What happens when we omit the join condition?

In [ ]:
conn.executescript('''

DROP TABLE IF EXISTS Coach_Q1;
DROP TABLE IF EXISTS Skater_Q1;
CREATE TABLE Coach_Q1 (cid INTEGER, name TEXT);
CREATE TABLE Skater_Q1 (aid INTEGER, name TEXT, cid INTEGER);

INSERT INTO Coach_Q1 VALUES (1, 'Dan'), (2, 'Don');
INSERT INTO Skater_Q1 VALUES (1, 'Tim', 2), (2, 'Tom', 1);
''')

# Run the tutorial query
query_q1 = '''
SELECT C.name AS cn, S.name AS sn
FROM Coach_Q1 C, Skater_Q1 S;
'''
print("Result of missing a join condition (Cartesian Product):")
table_part2_q1 = run_query(query_q1)

Result of missing a join condition (Cartesian Product):


In [ ]:
table_part2_q1

,cn,sn
0,Dan,Tim
1,Dan,Tom
2,Don,Tim
3,Don,Tom


---
### Part 4: Fixing SQL Q1 (Implicit vs. Explicit Joins)
The omission of a join condition leads to every row in table A multiplying with every row in table B.

To fix this, we need to link the primary key of the Coach table to the foreign key of the Skater table. Your lecture slides show how to do this in the `WHERE` clause (Implicit Join) . In the industry, however in practice, data analyst almost exclusively use the `JOIN ... ON` syntax (Explicit Join) for these kind of uses

In [ ]:
# 1. The WHERE Method (Implicit Inner Join from Slide 59)
query_where = '''
SELECT C.name AS Coach_Name, S.name AS Skater_Name
FROM Coach_Q1 C, Skater_Q1 S
WHERE C.cid = S.cid;
'''
print("Using WHERE (this is equivalent to INNER JOIN as well):")
display(run_query(query_where))

print("\n----------------------------------------\n")

# 2. The LEFT JOIN Method (Explicit Outer Join from Slide 61)
query_left = '''
SELECT C.name AS Coach_Name, S.name AS Skater_Name
FROM Coach_Q1 C
LEFT OUTER JOIN Skater_Q1 S ON C.cid = S.cid;
'''
print("Using LEFT JOIN (Dave is included with a None/NULL skater):")
display(run_query(query_left))



Using WHERE (this is equivalent to INNER JOIN as well):


,Coach_Name,Skater_Name
0,Dan,Tom
1,Don,Tim



----------------------------------------

Using LEFT JOIN (Dave is included with a None/NULL skater):


,Coach_Name,Skater_Name
0,Dan,Tom
1,Don,Tim


In [ ]:

conn.executescript('''

DROP TABLE IF EXISTS Coach_Q1;
DROP TABLE IF EXISTS Skater_Q1;
CREATE TABLE Coach_Q1 (cid INTEGER, name TEXT);
CREATE TABLE Skater_Q1 (aid INTEGER, name TEXT, cid INTEGER);


-- Notice Coach 3 (Dave) is being inserted here!
INSERT INTO Coach_Q1 VALUES (1, 'Dan'), (2, 'Don'), (3, 'Dave');
INSERT INTO Skater_Q1 VALUES (1, 'Tim', 2), (2, 'Tom', 1);


''')

In [ ]:
display(run_query(query_where))
display(run_query(query_left))

,Coach_Name,Skater_Name
0,Dan,Tom
1,Don,Tim


,Coach_Name,Skater_Name
0,Dan,Tom
1,Don,Tim
2,Dave,None


In [ ]:

# 3. The LEFT JOIN Method Let's exclude the OUTER
query_left_modified = '''
SELECT C.name AS Coach_Name, S.name AS Skater_Name
FROM Coach_Q1 C
LEFT JOIN Skater_Q1 S ON C.cid = S.cid;
'''
print("Using LEFT JOIN (Dave is included with a None/NULL skater):")
display(run_query(query_left_modified))

Using LEFT JOIN (Dave is included with a None/NULL skater):


,Coach_Name,Skater_Name
0,Dan,Tom
1,Don,Tim
2,Dave,None


You will see the difference between LEFT and WHERE/INNER joins once you start to consider null values

In [ ]:

# 4. INNER join
query_inner = '''
SELECT C.name AS Coach_Name, S.name AS Skater_Name
FROM Coach_Q1 C
INNER JOIN Skater_Q1 S ON C.cid = S.cid;
'''
print("Using INNER JOIN, same as WHERE:")
display(run_query(query_inner))

Using INNER JOIN, same as WHERE:


,Coach_Name,Skater_Name
0,Dan,Tom
1,Don,Tim


In [ ]:
display(run_query(query_where))

,Coach_Name,Skater_Name
0,Dan,Tom
1,Don,Tim


In [ ]:
display(run_query(query_left))

,Coach_Name,Skater_Name
0,Dan,Tom
1,Don,Tim
2,Dave,None


---
### Part 2: SQL Q2 - Complex Joins and Aggregation
We are asked to calculate the total monthly revenue generated by each country. Crucially, the report **must include all countries** listed in the Customers table, even if no customers in that country have an active subscription.

This requires:
1. `LEFT JOIN` (or `LEFT OUTER JOIN`): To ensure countries without matches are not dropped
2. `SUM()`: An aggregation function to total the revenue .
3. `GROUP BY`: To partition the tuples by country.

In [ ]:
# Setup richer tables for Q2 safely
conn.executescript('''
DROP TABLE IF EXISTS Subscriptions;
DROP TABLE IF EXISTS Plans;
DROP TABLE IF EXISTS Customers;

CREATE TABLE Customers (CID INTEGER PRIMARY KEY, name TEXT, country TEXT);
CREATE TABLE Plans (pid INTEGER PRIMARY KEY, planName TEXT, monthlyCost REAL);
CREATE TABLE Subscriptions (sid INTEGER PRIMARY KEY, cid INTEGER REFERENCES Customers(CID), pid INTEGER REFERENCES Plans(pid), isActive BOOLEAN);

-- Richer mock data
INSERT INTO Customers VALUES
    (1, 'Alice', 'Singapore'), (2, 'Bob', 'Singapore'), (3, 'Hannah', 'Singapore'),
    (4, 'Charlie', 'Japan'),
    (5, 'David', 'USA'), (6, 'Emma', 'USA'),
    (7, 'Fiona', 'UK'),
    (8, 'George', 'Australia');

INSERT INTO Plans VALUES
    (10, 'Basic', 10.00), (20, 'Pro', 20.00), (30, 'Enterprise', 50.00);

INSERT INTO Subscriptions VALUES
    (100, 1, 20, TRUE),   -- Alice (SG): Pro ($20)
    (101, 2, 30, TRUE),   -- Bob (SG): Enterprise ($50)
    (102, 3, 10, TRUE),   -- Hannah (SG): Basic ($10)
    (103, 4, 10, FALSE),  -- Charlie (JP): Basic (INACTIVE - $0)
    (104, 5, 20, TRUE),   -- David (USA): Pro ($20)
    (105, 6, 20, TRUE),   -- Emma (USA): Pro ($20)
    (106, 7, 10, TRUE),   -- Fiona (UK): Basic ($10)
    (107, 7, 20, FALSE),  -- Fiona (UK): Pro (INACTIVE - $0)
    (108, 8, 30, TRUE);   -- George (AUS): Enterprise ($50)
''')

# The Solution Query for Q2 [cite: 1089-1090, 1124-1126]
query_q2 = '''
SELECT c.country, SUM(p.monthlyCost) AS totalMonthlyRevenue
FROM Customers c
LEFT JOIN Subscriptions s ON c.CID = s.cid AND s.isActive = TRUE
LEFT JOIN Plans p ON s.pid = p.pid
GROUP BY c.country;
'''
print("--- Q2 Solution: Revenue by Country ---")
display(run_query(query_q2))

--- Q2 Solution: Revenue by Country ---


,country,totalMonthlyRevenue
0,Australia,50.0
1,Japan,NaN
2,Singapore,80.0
3,UK,10.0
4,USA,40.0


In [ ]:
# Step 1: Understanding the Left Join behavior
# Notice that every customer is present, even those without subscriptions.
# Their subscription details simply appear as None/Null.
query_step_1 = '''
SELECT c.name, c.country, s.sid
FROM Customers c
LEFT JOIN Subscriptions s ON c.CID = s.cid;
'''
print("--- Step 1: Customer List with Subscriptions ---")
display(run_query(query_step_1))

--- Step 1: Customer List with Subscriptions ---


,name,country,sid
0,Alice,Singapore,100
1,Bob,Singapore,101
2,Hannah,Singapore,102
3,Charlie,Japan,103
4,David,USA,104
5,Emma,USA,105
6,Fiona,UK,106
7,Fiona,UK,107
8,George,Australia,108


In [ ]:
# Step 2: Filtering for Active status during the Join
# Watch Charlie (Japan): His subscription is inactive, so the join "fails."
# Because it's a LEFT JOIN, Charlie stays in the list, but with NULLs.
query_step_2 = '''
SELECT c.name, c.country, s.isActive, p.monthlyCost
FROM Customers c
LEFT JOIN Subscriptions s ON c.CID = s.cid AND s.isActive = TRUE
LEFT JOIN Plans p ON s.pid = p.pid;
'''
print("--- Step 2: Filtering for Active Status ---")
display(run_query(query_step_2))

--- Step 2: Filtering for Active Status ---


,name,country,isActive,monthlyCost
0,Alice,Singapore,1.0,20.0
1,Bob,Singapore,1.0,50.0
2,Hannah,Singapore,1.0,10.0
3,Charlie,Japan,NaN,NaN
4,David,USA,1.0,20.0
5,Emma,USA,1.0,20.0
6,Fiona,UK,1.0,10.0
7,George,Australia,1.0,50.0


In [ ]:
# Step 3: The Final Calculation
# We group by country and sum the costs we isolated in Step 2.
query_step_3 = '''
SELECT c.country, SUM(p.monthlyCost) AS totalMonthlyRevenue
FROM Customers c
LEFT JOIN Subscriptions s ON c.CID = s.cid AND s.isActive = TRUE
LEFT JOIN Plans p ON s.pid = p.pid
GROUP BY c.country;
'''
print("--- Step 3: Final Revenue by Country ---")
display(run_query(query_step_3))

--- Step 3: Final Revenue by Country ---


,country,totalMonthlyRevenue
0,Australia,50.0
1,Japan,NaN
2,Singapore,80.0
3,UK,10.0
4,USA,40.0


In [ ]:
print("\n--- Additional Goal 1: Filtering Groups with HAVING ---")
# Let's find only the countries generating $30 or more in active revenue
query_having = '''
SELECT c.country, SUM(p.monthlyCost) AS totalMonthlyRevenue
FROM Customers c
LEFT JOIN Subscriptions s ON c.CID = s.cid AND s.isActive = TRUE
LEFT JOIN Plans p ON s.pid = p.pid
GROUP BY c.country
HAVING SUM(p.monthlyCost) >= 30;
'''
display(run_query(query_having))


print("\n--- Additional Goal 2: Case Analysis with UNION ---")
# Mimicking Slide 72: Using UNION to assign classes based on a condition
query_union = '''
SELECT planName, monthlyCost, 'Premium Tier' AS PlanClass
FROM Plans
WHERE monthlyCost >= 20
UNION ALL
SELECT planName, monthlyCost, 'Standard Tier' AS PlanClass
FROM Plans
WHERE monthlyCost < 20;
'''
display(run_query(query_union))


--- Additional Goal 1: Filtering Groups with HAVING ---


,country,totalMonthlyRevenue
0,Australia,50.0
1,Singapore,80.0
2,USA,40.0



--- Additional Goal 2: Case Analysis with UNION ---


,planName,monthlyCost,PlanClass
0,Pro,20.0,Premium Tier
1,Enterprise,50.0,Premium Tier
2,Basic,10.0,Standard Tier
